# LAB 7 — Avaliação Contínua de RAG com LangFuse
## Aula 5 — Avaliação de Pipelines RAG

Explora as **capacidades avançadas do LangFuse** para fazer **avaliação contínua** do RAG em produção, indo além do envio simples de scores do LAB 3:

1. **Datasets versionados** — dataset oficial de avaliação no LangFuse, com versões
2. **Dataset Runs (Experimentos)** — executa uma versão do pipeline contra o dataset e gera um run comparável
3. **Custom Scores** — score numérico (RAGAS), categórico (categoria de erro) e boolean (passou/falhou)
4. **Sessions & Users** — agrupa traces por usuário/conversa para análise comportamental
5. **Tags & Filtros** — categoriza traces e busca por metadata/score/tag
6. **Annotation Queue** — fila de revisão humana para traces com baixa qualidade
7. **Prompt Management** — versiona prompts dentro do LangFuse com rollback
8. **Drift Detection** — compara médias entre janelas temporais (semana atual vs anterior)
9. **`@observe` Decorator + LangfuseCallbackHandler** — instrumentação automática em produção
10. **Cron Job / Schedule** — padrão de avaliação contínua agendada

**Stack:** Groq + Ollama BGE-M3 + OpenSearch (mesma da Aula 5).

**Pré-requisito:** LAB 1, 2 e 3 executados.


## 1. Instalação + Stack


In [ ]:
import subprocess, sys
for pkg in ['ragas>=0.1.16','datasets','pandas','matplotlib','seaborn','tqdm',
            'langchain>=0.2','langchain-core>=0.2',
            'langchain-groq>=0.1','langchain-ollama>=0.1',
            'opensearch-py>=2.7','python-dotenv>=1.0','langfuse>=2.36']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
print('dependencias instaladas')


In [ ]:
import os, json, time, warnings, math, statistics, datetime
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

for ep in [Path.cwd()/'.env', Path.home()/'mba-rag'/'.env',
            Path.cwd().parent.parent/'.env']:
    if ep.exists(): load_dotenv(ep, override=True); break

GROQ_API_KEY       = os.getenv('GROQ_API_KEY','')
GROQ_LLM_MODEL     = os.getenv('GROQ_LLM_MODEL','llama-3.1-8b-instant')
OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL','http://localhost:11434')
OLLAMA_LLM_MODEL   = os.getenv('OLLAMA_LLM_MODEL','llama3.2:3b')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL','bge-m3')
OPENSEARCH_HOST    = os.getenv('OPENSEARCH_HOST','localhost')
OPENSEARCH_PORT    = int(os.getenv('OPENSEARCH_PORT',9200))
OPENSEARCH_USER    = os.getenv('OPENSEARCH_USER','admin')
OPENSEARCH_PASS    = os.getenv('OPENSEARCH_PASSWORD','admin')
INDEX_NAME         = os.getenv('INDEX_NAME','corpus_juridico_aula4')
LANGFUSE_HOST      = os.getenv('LANGFUSE_HOST','http://localhost:3000')
LANGFUSE_PK        = os.getenv('LANGFUSE_PUBLIC_KEY','pk-lf-xxxx')
LANGFUSE_SK        = os.getenv('LANGFUSE_SECRET_KEY','sk-lf-xxxx')

META = {'faithfulness':0.80, 'answer_relevancy':0.75,
        'context_recall':0.70, 'context_precision':0.70}

from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from opensearchpy import OpenSearch
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langfuse import Langfuse

# LLM
llm = None; LLM_PROVIDER = None; LLM_MODEL = None
if GROQ_API_KEY:
    try:
        c = ChatGroq(model=GROQ_LLM_MODEL, api_key=GROQ_API_KEY,
                     temperature=0.0, max_tokens=512, timeout=30)
        c.invoke([HumanMessage(content='ok')])
        llm, LLM_PROVIDER, LLM_MODEL = c, 'groq', GROQ_LLM_MODEL
    except Exception: pass
if llm is None:
    llm = ChatOllama(model=OLLAMA_LLM_MODEL, base_url=OLLAMA_BASE_URL,
                     temperature=0.0, num_predict=512)
    LLM_PROVIDER, LLM_MODEL = 'ollama', OLLAMA_LLM_MODEL

embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL, base_url=OLLAMA_BASE_URL)
_ = embeddings.embed_query('teste'); assert len(_) == 1024

os_client = OpenSearch(
    hosts=[{'host':OPENSEARCH_HOST,'port':OPENSEARCH_PORT}],
    http_auth=(OPENSEARCH_USER,OPENSEARCH_PASS),
    use_ssl=False, verify_certs=False, timeout=60,
)

lf = Langfuse(public_key=LANGFUSE_PK, secret_key=LANGFUSE_SK, host=LANGFUSE_HOST)
try:
    lf.auth_check(); LANGFUSE_OK = True
    print(f'LangFuse OK em {LANGFUSE_HOST}')
except Exception as e:
    LANGFUSE_OK = False
    print(f'LangFuse indisponivel: {e} - lab demonstrativo')

ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)
print(f'Stack OK | LLM={LLM_PROVIDER}/{LLM_MODEL} | embed={OLLAMA_EMBED_MODEL} | index={INDEX_NAME}')


## 2. Dataset Versionado no LangFuse

LangFuse permite registrar um **dataset oficial de avaliação** com versionamento. Cada `dataset_item` representa um par QA com ground-truth. Pipelines podem então ser executados contra o dataset gerando **Runs** comparáveis na UI.

A vantagem: o dataset deixa de ser um arquivo local e passa a ser um **artefato versionado e compartilhado** entre equipes.


In [ ]:
DATASET_NAME = 'aula5-juridico-tcu-eval'

# Carrega o dataset do LAB 1
with open('dataset_avaliacao_completo.json', encoding='utf-8') as f:
    dataset_local = json.load(f)

if LANGFUSE_OK:
    # Cria (ou recupera) o dataset no LangFuse
    try:
        lf.create_dataset(
            name=DATASET_NAME,
            description='50 pares QA juridicos TCU 2026 com ground-truth - Aula 5 MBA',
            metadata={'aula':'5','dominio':'juridico','versao':'1.0',
                      'fonte':'corpus_avaliacao_aula5.json'},
        )
        print(f'dataset criado: {DATASET_NAME}')
    except Exception as e:
        print(f'dataset ja existe ou erro: {e}')

    # Sobe os items (idempotente quando se reutiliza mesmo id)
    SUBSET = 20  # primeiros 20 pares para o exemplo
    for par in dataset_local[:SUBSET]:
        lf.create_dataset_item(
            dataset_name=DATASET_NAME,
            input={'question': par['question']},
            expected_output={'answer': par['ground_truth']},
            metadata={'id': par.get('id',''),
                      'tipo': par.get('tipo',''),
                      'dificuldade': par.get('dificuldade','')},
        )
    lf.flush()
    print(f'{SUBSET} dataset_items enviados ao LangFuse')
    print(f'Veja em: {LANGFUSE_HOST}/datasets/{DATASET_NAME}')
else:
    print('LangFuse offline - pulando upload do dataset')


## 3. Dataset Run (Experimento)

Um **Run** executa uma versão do pipeline contra o dataset e cria N traces vinculados. Permite comparar `naive_rag_v1` vs `advanced_rag_v2` lado a lado na UI do LangFuse.


In [ ]:
PROMPT_RAG = ChatPromptTemplate.from_messages([
    ('system','Voce e um assistente juridico brasileiro. Responda APENAS com base nos trechos.'),
    ('human','TRECHOS:\n{contextos}\n\nPERGUNTA: {pergunta}\n\nRESPOSTA:')
])
RAG_CHAIN = PROMPT_RAG | llm | StrOutputParser()

def naive_rag_pipeline(question: str, k: int = 5):
    v = embeddings.embed_query(question)
    r = os_client.search(index=INDEX_NAME,
                          body={'size':k,
                                'query':{'knn':{'embedding':{'vector':v,'k':k}}},
                                '_source':['titulo','conteudo']})
    ctxs = [(h['_source'].get('titulo','')+'\n'+h['_source'].get('conteudo','')).strip()
             for h in r['hits']['hits']]
    bloco = '\n\n'.join(f'[Doc {i+1}]\n{c[:600]}' for i, c in enumerate(ctxs))
    ans = RAG_CHAIN.invoke({'contextos':bloco,'pergunta':question}).strip()
    return ans, ctxs

if LANGFUSE_OK:
    # Run name e metadata (versionado!)
    RUN_NAME = f'naive_rag_v1_{LLM_PROVIDER}_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}'
    RUN_DESC = f'Naive RAG (k=5, BM25+kNN) com {LLM_PROVIDER}/{LLM_MODEL}'

    items = lf.get_dataset(DATASET_NAME).items
    print(f'Executando run "{RUN_NAME}" em {len(items)} itens...')

    from tqdm.auto import tqdm
    for item in tqdm(items, desc='dataset run'):
        question = item.input.get('question','')
        gt = (item.expected_output or {}).get('answer','')
        # link() cria um trace vinculado ao item E ao run
        with item.observe(run_name=RUN_NAME, run_description=RUN_DESC,
                           run_metadata={'pipeline':'naive_rag_v1',
                                         'llm_provider': LLM_PROVIDER,
                                         'llm_model': LLM_MODEL,
                                         'embed_model': OLLAMA_EMBED_MODEL,
                                         'index': INDEX_NAME, 'k': 5}) as root_trace:
            ans, ctxs = naive_rag_pipeline(question)
            # adiciona dados no trace raiz
            root_trace.update(output={'answer': ans, 'n_contextos': len(ctxs)})
    lf.flush()
    print(f'run finalizado: {LANGFUSE_HOST}/datasets/{DATASET_NAME}/runs')
else:
    print('LangFuse offline - pulando run')


## 4. Custom Scores — Numéricos, Categóricos e Boolean

A Scores API aceita 3 tipos de score:
- **NUMERIC** — RAGAS, MRR, latência (float)
- **CATEGORICAL** — categoria de erro (`retrieval_ruim` / `prompt_vago` / `query_ambigua`)
- **BOOLEAN** — passa/falha por meta

Combinar os 3 dá **alta riqueza analítica** nos dashboards.


In [ ]:
# Simula um pipeline com avaliacao multi-score em 5 traces novos
from ragas import evaluate
from datasets import Dataset
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

if LANGFUSE_OK:
    print('Gerando 5 traces com 4 numericos + 1 categorico + 1 boolean cada...')
    for par in dataset_local[:5]:
        trace = lf.trace(
            name='RAG-com-multi-scores',
            input={'question': par['question']},
            metadata={'lab':'LAB7','pipeline':'naive_rag_v1'},
            tags=['lab7','multi-score','juridico'],
        )
        ans, ctxs = naive_rag_pipeline(par['question'])
        trace.update(output={'answer': ans})

        # avalia com RAGAS
        ds1 = Dataset.from_dict({'question':[par['question']],
                                   'answer':[ans],
                                   'contexts':[ctxs],
                                   'ground_truth':[par['ground_truth']]})
        res = evaluate(ds1, metrics=[faithfulness, answer_relevancy,
                                       context_recall, context_precision],
                        llm=ragas_llm, embeddings=ragas_embeddings,
                        raise_exceptions=False)
        scores_num = {m: float(res[m]) if not math.isnan(float(res[m])) else 0.0
                       for m in META}

        # 4 numericos (RAGAS)
        for n, v in scores_num.items():
            lf.score(trace_id=trace.id, name=n, value=v, data_type='NUMERIC',
                      comment=f'meta {META[n]}')

        # 1 categorico (faixa de qualidade global)
        media = statistics.mean(scores_num.values())
        if   media >= 0.80: cat = 'excelente'
        elif media >= 0.65: cat = 'aceitavel'
        elif media >= 0.50: cat = 'marginal'
        else:               cat = 'insuficiente'
        lf.score(trace_id=trace.id, name='qualidade_geral',
                  value=cat, data_type='CATEGORICAL',
                  comment=f'media {media:.3f}')

        # 1 boolean (passa todas as metas?)
        passa = all(scores_num[m] >= META[m] for m in META)
        lf.score(trace_id=trace.id, name='passa_quality_gate',
                  value=passa, data_type='BOOLEAN',
                  comment='True quando todas as 4 metas RAGAS sao atingidas')

        print(f'  trace {trace.id} | media={media:.3f} | cat={cat} | passa={passa}')
    lf.flush()
    print('\n5 traces multi-score gravados.')
else:
    print('LangFuse offline')


## 5. Sessions & Users — Agrupando Conversas

Em RAG conversacional (Aula 4 EXEMPLO 3), múltiplos turnos pertencem à **mesma sessão de um usuário**. LangFuse permite vincular traces a:
- **session_id**: id da conversa multi-turno
- **user_id**: id do usuário final (mascarado em produção)

Isso possibilita análises como *latência média por usuário* e *evolução do Faithfulness por sessão*.


In [ ]:
if LANGFUSE_OK:
    # simula 1 sessao com 3 turnos
    SESSION = f'sess-juridico-{datetime.datetime.now().strftime("%Y%m%d%H%M%S")}'
    USER    = 'tornis@tornis.com.br'
    perguntas_sess = [
        'O que e operacao de credito com garantia da Uniao?',
        'Quais irregularidades costumam ser apuradas pelo TCU nessas operacoes?',
        'Quando o TCU pode determinar medida cautelar?',
    ]
    print(f'Sessao: {SESSION} | usuario: {USER}')
    for i, q in enumerate(perguntas_sess, 1):
        tr = lf.trace(
            name='conversational-rag-turn',
            session_id=SESSION, user_id=USER,
            input={'turn': i, 'question': q},
            metadata={'lab':'LAB7','feature':'sessions','turn': i},
            tags=['conversational','session','lab7'],
        )
        ans, ctxs = naive_rag_pipeline(q)
        tr.update(output={'turn': i, 'answer': ans})
        lf.score(trace_id=tr.id, name='turn_number', value=i, data_type='NUMERIC')
        print(f'  turn {i}: {q[:60]}... -> {ans[:60]}...')
    lf.flush()
    print(f'\nveja por sessao: {LANGFUSE_HOST}/sessions/{SESSION}')
    print(f'veja por usuario: {LANGFUSE_HOST}/users')
else:
    print('LangFuse offline')


## 6. Tags & Filtros via API

LangFuse permite **buscar traces** por tag, score, metadata, intervalo de tempo. Útil para construir *cohorts* de análise.


In [ ]:
if LANGFUSE_OK:
    try:
        # busca os 10 traces mais recentes com tag 'lab7'
        recents = lf.fetch_traces(tags=['lab7'], limit=10)
        items = getattr(recents, 'data', recents)
        print(f'traces com tag "lab7" recentes: {len(items)}')
        for t in items[:5]:
            tid = getattr(t, 'id', '?')
            tname = getattr(t, 'name', '')
            print(f'  {tid[:8]}... | {tname}')
    except Exception as e:
        print(f'fetch_traces nao disponivel ({e}). Use a UI:')
        print(f'  {LANGFUSE_HOST}/traces?tags=lab7')
else:
    print('LangFuse offline')


## 7. Annotation Queue — Revisão Humana

Para queries com Faithfulness baixo (< 0.70), enviamos o trace para uma **fila de anotação humana** no LangFuse. Especialistas jurídicos revisam e atribuem scores manualmente, que então alimentam o treinamento de evaluators customizados.


In [ ]:
# Padrao: marca traces com tag 'review-needed' e adiciona comentario
# explicando o motivo. Na UI a equipe filtra por essa tag para revisar.

THRESHOLD_REVIEW = 0.70

if LANGFUSE_OK:
    print(f'Marcando para revisao: traces com faithfulness < {THRESHOLD_REVIEW}')
    flagged = 0
    for par in dataset_local[:5]:
        tr = lf.trace(
            name='rag-com-review-flag',
            input={'question': par['question']},
            metadata={'lab':'LAB7','feature':'annotation-queue'},
            tags=['lab7','rag'],
        )
        ans, ctxs = naive_rag_pipeline(par['question'])
        tr.update(output={'answer': ans})
        # avaliacao
        ds1 = Dataset.from_dict({'question':[par['question']],'answer':[ans],
                                  'contexts':[ctxs],'ground_truth':[par['ground_truth']]})
        res = evaluate(ds1, metrics=[faithfulness],
                        llm=ragas_llm, embeddings=ragas_embeddings,
                        raise_exceptions=False)
        faith = float(res['faithfulness']) if not math.isnan(float(res['faithfulness'])) else 0.0
        lf.score(trace_id=tr.id, name='faithfulness', value=faith,
                  data_type='NUMERIC')

        if faith < THRESHOLD_REVIEW:
            # adiciona tag adicional 'review-needed' atualizando o trace
            tr.update(tags=['lab7','rag','review-needed'],
                       metadata={'review_reason': f'faithfulness {faith:.3f} < {THRESHOLD_REVIEW}'})
            flagged += 1
            print(f'  FLAG {tr.id[:8]}... faith={faith:.3f} -> review-needed')
        else:
            print(f'   OK  {tr.id[:8]}... faith={faith:.3f}')
    lf.flush()
    print(f'\n{flagged} traces marcados para revisao humana.')
    print(f'Acesse: {LANGFUSE_HOST}/traces?tags=review-needed')
else:
    print('LangFuse offline')


## 8. Prompt Management (Versionamento de Prompts)

LangFuse pode hospedar e versionar **prompts em produção**. Mudanças de prompt ficam rastreáveis e linkadas aos traces — viabilizando rollback rápido em caso de regressão.


In [ ]:
PROMPT_NAME = 'aula5-rag-juridico'

if LANGFUSE_OK:
    try:
        # cria/atualiza um prompt versionado (text prompt)
        lf.create_prompt(
            name=PROMPT_NAME,
            prompt=('Voce e um assistente juridico brasileiro especializado em '
                    'acordaos do TCU. Responda APENAS com base nos trechos abaixo. '
                    'Se a informacao nao estiver nos trechos, diga claramente.\n\n'
                    'TRECHOS:\n{{contextos}}\n\n'
                    'PERGUNTA: {{pergunta}}\n\n'
                    'RESPOSTA:'),
            labels=['production','aula5','lab7'],
            tags=['rag','juridico','tcu'],
        )
        print(f'prompt "{PROMPT_NAME}" criado/atualizado')

        # busca a versao production
        p = lf.get_prompt(PROMPT_NAME, label='production')
        print(f'\nversao atual: {p.version}')
        print(f'labels: {p.labels}')
        print(f'\ncontent (head):')
        print(p.prompt[:240] + '...')
        print(f'\nveja em: {LANGFUSE_HOST}/prompts/{PROMPT_NAME}')
    except Exception as e:
        print(f'erro prompt mgmt: {e}')
else:
    print('LangFuse offline')


## 9. Drift Detection — Comparação Temporal

**Drift** = mudança estatística na qualidade do sistema ao longo do tempo. LangFuse expõe scores agregados via API; comparar **janela atual vs janela anterior** detecta degradação.


In [ ]:
# Simulacao: usa scores agregados de runs ja gravados ou gera dados sinteticos.
# Em producao real: lf.api.score.list(name='faithfulness', from=..., to=...).

import random
random.seed(42)

# Simula 30 dias de medias diarias de faithfulness
hoje = datetime.date.today()
dias = [hoje - datetime.timedelta(days=i) for i in range(30, 0, -1)]
# Drift simulado: semanas mais recentes ficam piores
faith_diario = []
for i, d in enumerate(dias):
    base = 0.82 if i < 21 else 0.74  # drift na 4a semana
    faith_diario.append(base + random.uniform(-0.05, 0.05))

semana_atual    = faith_diario[-7:]
semana_anterior = faith_diario[-14:-7]
duas_semanas_atras = faith_diario[-21:-14]

media_atual = statistics.mean(semana_atual)
media_ant   = statistics.mean(semana_anterior)
media_2     = statistics.mean(duas_semanas_atras)
delta_wow   = media_atual - media_ant

print('=== Drift Detection (Faithfulness) ===')
print(f'  2 semanas atras  : {media_2:.4f}')
print(f'  semana anterior  : {media_ant:.4f}')
print(f'  semana atual     : {media_atual:.4f}')
print(f'  delta WoW        : {delta_wow:+.4f}')
print(f'  alerta           : {"SIM - investigar" if delta_wow < -0.05 else "OK"}')

# Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dias, faith_diario, marker='o', color='#3498db', label='Faithfulness diario')
ax.axhline(0.80, color='red', linestyle='--', label='meta 0.80')
ax.axvspan(dias[-7], dias[-1], color='orange', alpha=0.2, label='semana atual')
ax.set_ylim(0.5, 1.0); ax.set_ylabel('Faithfulness medio'); ax.set_xlabel('Data')
ax.set_title('Drift Detection - 30 dias de Faithfulness', fontweight='bold')
ax.grid(True, alpha=0.3); ax.legend()
plt.xticks(rotation=45); plt.tight_layout()
plt.savefig('lab7_drift_detection.png', dpi=140); plt.show()
print('\nlab7_drift_detection.png salvo')

# se LangFuse OK, registra o alerta como um score num trace de monitoramento
if LANGFUSE_OK:
    tr = lf.trace(
        name='monitoring-drift-check',
        metadata={'lab':'LAB7','feature':'drift-detection',
                  'analise':'WoW'},
        tags=['monitoring','drift','lab7'],
    )
    lf.score(trace_id=tr.id, name='faithfulness_semana_atual', value=media_atual,
              data_type='NUMERIC')
    lf.score(trace_id=tr.id, name='faithfulness_semana_anterior', value=media_ant,
              data_type='NUMERIC')
    lf.score(trace_id=tr.id, name='delta_WoW', value=delta_wow,
              data_type='NUMERIC',
              comment='abaixo de -0.05 dispara alerta')
    lf.score(trace_id=tr.id, name='drift_alert',
              value=(delta_wow < -0.05),
              data_type='BOOLEAN',
              comment='True quando delta WoW < -0.05')
    lf.flush()
    print(f'trace de monitoramento: {LANGFUSE_HOST}/traces/{tr.id}')


## 10. `@observe` + LangfuseCallbackHandler

O decorator `@observe` instrumenta funções automaticamente. Combinado com o **CallbackHandler** da LangChain, captura chamadas de LLM, embeddings e retrievers sem código extra — ideal para serviços de produção.


In [ ]:
try:
    from langfuse.decorators import observe, langfuse_context
    from langfuse.callback import CallbackHandler
    OBS = True
except Exception as e:
    print(f'decorators indisponiveis: {e}')
    OBS = False

if OBS and LANGFUSE_OK:
    # CallbackHandler captura cada chamada LangChain (LLM, embeddings, retrievers)
    cb = CallbackHandler(public_key=LANGFUSE_PK, secret_key=LANGFUSE_SK,
                          host=LANGFUSE_HOST)

    @observe(name='rag-producao-com-callback')
    def rag_producao(question: str) -> dict:
        langfuse_context.update_current_observation(
            metadata={'lab':'LAB7','feature':'observe+callback'},
            tags=['producao','observe','lab7'],
        )
        # passar callbacks para a cadeia faz o CallbackHandler criar spans
        v = embeddings.embed_query(question)
        r = os_client.search(index=INDEX_NAME,
                              body={'size':5,
                                    'query':{'knn':{'embedding':{'vector':v,'k':5}}},
                                    '_source':['titulo','conteudo']})
        ctxs = [(h['_source'].get('titulo','')+'\n'+h['_source'].get('conteudo','')).strip()
                 for h in r['hits']['hits']]
        bloco = '\n\n'.join(f'[Doc {i+1}]\n{c[:500]}' for i, c in enumerate(ctxs))
        ans = RAG_CHAIN.invoke({'contextos':bloco,'pergunta':question},
                                config={'callbacks':[cb]}).strip()
        return {'answer': ans, 'n_ctxs': len(ctxs),
                'trace_id': langfuse_context.get_current_trace_id()}

    out = rag_producao(dataset_local[0]['question'])
    print(f'\ntrace_id (@observe): {out["trace_id"]}')
    print(f'answer head: {out["answer"][:160]}...')
    print(f'\nVeja spans automaticos em: {LANGFUSE_HOST}/traces/{out["trace_id"]}')
else:
    print('@observe ou LangFuse offline - exemplo apenas documentado')


## 11. Cron Job — Padrão de Avaliação Contínua Agendada

Em produção, agende este pipeline (cron ou Airflow) para rodar **toda noite**: amostra N traces das últimas 24h, avalia com RAGAS, registra agregados no LangFuse e dispara alerta se a média cair abaixo da meta.


In [ ]:
def evaluate_recent_traces(amostra: int = 20, since_hours: int = 24):
    """Padrao para job agendado.

    Em producao real, troque dataset_local[:amostra] por
    lf.api.trace.list(from=...,limit=amostra) e avalie cada um.
    """
    if not LANGFUSE_OK:
        print('LangFuse offline - simulando')

    scores_globais = {m: [] for m in META}
    for par in dataset_local[:amostra]:
        ans, ctxs = naive_rag_pipeline(par['question'])
        ds1 = Dataset.from_dict({'question':[par['question']],
                                   'answer':[ans],
                                   'contexts':[ctxs],
                                   'ground_truth':[par['ground_truth']]})
        res = evaluate(ds1, metrics=[faithfulness, answer_relevancy,
                                       context_recall, context_precision],
                        llm=ragas_llm, embeddings=ragas_embeddings,
                        raise_exceptions=False)
        for m in META:
            v = float(res[m])
            if not math.isnan(v): scores_globais[m].append(v)

    medias = {m: statistics.mean(scores_globais[m]) for m in META}
    print('\n=== Job de monitoramento RAGAS (ultimas 24h) ===')
    for m, target in META.items():
        med = medias[m]
        alerta = ' ALERTA' if med < target else ''
        print(f'  {m:<22} {med:.4f} (meta {target}){alerta}')

    if LANGFUSE_OK:
        tr = lf.trace(
            name='daily-ragas-monitoring',
            metadata={'lab':'LAB7','schedule':'daily','amostra':amostra,
                      'since_hours': since_hours},
            tags=['monitoring','daily','lab7'],
        )
        for m, v in medias.items():
            lf.score(trace_id=tr.id, name=f'daily_{m}', value=v,
                      data_type='NUMERIC',
                      comment=f'media diaria | meta {META[m]}')
            lf.score(trace_id=tr.id, name=f'daily_{m}_alert',
                      value=(v < META[m]), data_type='BOOLEAN')
        lf.flush()
        print(f'monitoring trace: {LANGFUSE_HOST}/traces/{tr.id}')
    return medias

# rodar 1x manualmente (em producao seria via cron)
_medias = evaluate_recent_traces(amostra=5, since_hours=24)


## 12. Resumo — Capacidades LangFuse Cobertas

| Capacidade | Seção | Uso prático |
|------------|-------|-------------|
| Datasets versionados | §2 | Ground-truth oficial e compartilhado |
| Dataset Runs / Experimentos | §3 | Comparar versões de pipeline lado a lado |
| Custom Scores (NUMERIC/CATEG/BOOL) | §4 | Riqueza de análise no dashboard |
| Sessions & Users | §5 | Análise por conversa e por usuário |
| Tags & Filtros API | §6 | Cohorts e queries reutilizáveis |
| Annotation Queue | §7 | Revisão humana de queries de baixa qualidade |
| Prompt Management | §8 | Versionamento e rollback de prompts em produção |
| Drift Detection | §9 | Comparação temporal e alertas WoW |
| `@observe` + CallbackHandler | §10 | Instrumentação automática zero-boilerplate |
| Cron Job de Monitoramento | §11 | Avaliação contínua agendada |

## Próximos Passos

1. Configurar **alertas automáticos** (Slack/Email) quando `drift_alert == True`
2. Treinar um **evaluator customizado** com os scores manuais da Annotation Queue
3. Integrar com **Airflow / GitHub Actions** para rodar o cron job em CI
4. Versionar o `dataset` a cada release de modelo (LangFuse versiona automaticamente)

## Referências

LANGFUSE. **Datasets and Experiments**. <https://langfuse.com/docs/datasets>.  
LANGFUSE. **Scores — Custom Scores**. <https://langfuse.com/docs/scores/custom>.  
LANGFUSE. **Sessions & Users**. <https://langfuse.com/docs/tracing-features/sessions>.  
LANGFUSE. **Prompt Management**. <https://langfuse.com/docs/prompts/get-started>.  
LANGFUSE. **Annotation queues**. <https://langfuse.com/docs/scores/annotation>.  
LANGFUSE. **`@observe` decorator / CallbackHandler**. <https://langfuse.com/docs/sdk/python/decorators>.  
ES, S. et al. **RAGAS**. arXiv:2309.15217, 2023.  
GROQ INC. **Groq API**. <https://console.groq.com/docs>.
